# ENGG2112: APTOS five-grade model training

This separate notebook trains an ImageNet-pretrained ResNet-18 baseline for grades 0–4, selects a checkpoint using validation macro-F1, and reports scikit-learn metrics. It consumes the prepared, versioned handoff; it does **not** download Kaggle data or create a new split.

**Status:** workflow only. This workspace currently lacks the prepared images and split manifest, so no model has been trained and no performance results are available. Do not present placeholder or prior audit figures as model results.

## Obtain the data first

Sign in to Kaggle, open the [APTOS 2019 competition data page](https://www.kaggle.com/competitions/aptos2019-blindness-detection/data), review and accept its rules, and download the authorized files yourself. Kaggle requires an account and rule acceptance; this notebook never requests credentials or downloads the dataset.

For raw files, put `train.csv` at `data/train.csv` and the PNGs in `data/train_images/`. Run the data-engineering notebook in `build` mode using a new run name, then set `DATA_ROOT` below to that generated handoff folder (for example `artifacts/notebook_runs/notebook_run_001`). Alternatively, use the matching prepared private handoff. The required handoff paths are `artifacts/splits/split_manifest.csv` and `data/processed_224_v1/`.

Install the project requirements plus PyTorch and TorchVision in the selected notebook kernel. On Apple Silicon, use the official PyTorch installation instructions for your macOS/Python version; pretrained ImageNet weights may need to download on first run.

In [1]:
from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd
from PIL import Image

SEED = 2112
DATA_ROOT = Path.cwd() / 'artifacts/notebook_runs/notebook_run_001'
OUTPUT_DIR = Path.cwd() / 'outputs' / 'aptos_resnet18_run'
BATCH_SIZE = 32
MAX_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # Reliable default for macOS notebooks.
CONFIRM_REVIEWED_FROZEN_SPLIT = False  # Set True only after team review and split freeze.
TEST_EVALUATION_COMPLETED = False

random.seed(SEED)
np.random.seed(SEED)

split_path = DATA_ROOT / 'artifacts/splits/split_manifest.csv'
prepared_dir = DATA_ROOT / 'data/processed_224_v1'
prepared_manifest_path = prepared_dir / 'processed_manifest.csv'
config_path = prepared_dir / 'preprocessing_config.json'
missing = [str(p) for p in (split_path, prepared_manifest_path, config_path) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        'Prepared APTOS handoff not found. Download authorized data from Kaggle, '
        'build or extract the matching handoff, then set DATA_ROOT. Missing: ' + ', '.join(missing)
    )

try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    from torchvision import models, transforms
    from torchvision.models import ResNet18_Weights
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                                 confusion_matrix, cohen_kappa_score, f1_score)
except ImportError as exc:
    raise ImportError(
        'Install the notebook-kernel dependencies: project requirements, PyTorch, '
        'TorchVision, and scikit-learn. Select that environment as this notebook kernel.'
    ) from exc

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('Device:', DEVICE, '| PyTorch:', torch.__version__, '| Handoff:', DATA_ROOT)

Device: mps | PyTorch: 2.14.1 | Handoff: /Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/artifacts/notebook_runs/notebook_run_001


## Model and selection plan

The baseline fine-tunes all ResNet-18 layers, replaces the final layer with five logits, and uses ImageNet normalization. Training-only augmentation is a horizontal flip and a small rotation. Cross-entropy class weights are calculated from the training split only. AdamW uses a small learning rate and weight decay.

After each epoch, select the checkpoint with the highest validation macro-F1; validation quadratic weighted kappa is reported as a secondary ordinal measure. Early stopping uses validation macro-F1. The test split is not used for training or selection. Test evaluation remains disabled unless the reviewed split is explicitly confirmed frozen below.

In [2]:
split = pd.read_csv(split_path, dtype={'id_code': str, 'group_id': str, 'pixel_sha256': str})
prepared = pd.read_csv(prepared_manifest_path, dtype={'id_code': str})
required_split = {'id_code', 'diagnosis', 'split', 'group_id', 'pixel_sha256'}
required_prepared = {'id_code', 'diagnosis', 'split', 'relative_path'}
if not required_split <= set(split.columns) or not required_prepared <= set(prepared.columns):
    raise ValueError('The handoff manifests do not match the expected data contract.')
for name, frame in [('split', split), ('prepared', prepared)]:
    if frame.id_code.isna().any() or frame.id_code.duplicated().any():
        raise ValueError(f'{name} manifest has a missing or duplicated image ID.')
    if not pd.to_numeric(frame.diagnosis, errors='coerce').isin(range(5)).all():
        raise ValueError(f'{name} manifest has a grade outside 0..4.')
    if not frame.split.isin(['train', 'validation', 'test']).all():
        raise ValueError(f'{name} manifest has an invalid split name.')
if set(split.id_code) != set(prepared.id_code):
    raise ValueError('Split and prepared manifests contain different image IDs.')
joined = split.merge(prepared, on='id_code', validate='one_to_one', suffixes=('', '_prepared'))
if not joined.diagnosis.eq(joined.diagnosis_prepared).all() or not joined.split.eq(joined.split_prepared).all():
    raise ValueError('Split and prepared manifests disagree on labels or assignments.')
for column in ('group_id', 'pixel_sha256'):
    if split.groupby(column).split.nunique().max() != 1:
        raise ValueError(f'{column} crosses split boundaries.')
if not joined.relative_path.eq(joined.id_code + '.png').all():
    raise ValueError('Expected prepared image filenames to be ID.png.')
preprocessing = json.loads(config_path.read_text(encoding='utf-8'))
if preprocessing.get('size') != 224 or preprocessing.get('mode') != 'RGB':
    raise ValueError('This baseline requires the prepared 224x224 RGB handoff.')
joined['image_path'] = [str(prepared_dir / name) for name in joined.relative_path]
missing_images = [p for p in joined.image_path if not Path(p).is_file()]
if missing_images:
    raise FileNotFoundError(f'{len(missing_images)} prepared images are missing; extract the complete handoff.')
joined['diagnosis'] = joined.diagnosis.astype(int)
train_frame = joined.loc[joined.split.eq('train')].reset_index(drop=True)
validation_frame = joined.loc[joined.split.eq('validation')].reset_index(drop=True)
test_frame = joined.loc[joined.split.eq('test')].reset_index(drop=True)
if train_frame.empty or validation_frame.empty or test_frame.empty:
    raise ValueError('Train, validation, and test splits must all be non-empty.')
print('Rows:', len(joined), '| train:', len(train_frame), '| validation:', len(validation_frame), '| test locked:', len(test_frame))
display(pd.crosstab(joined.split, joined.diagnosis).reindex(columns=range(5), fill_value=0))

Rows: 3600 | train: 2521 | validation: 538 | test locked: 541


diagnosis,0,1,2,3,4
split,,,,,
test,273,53,144,27,44
train,1258,253,684,127,199
validation,270,54,148,26,40


In [3]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FundusDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as image:
            image = image.convert('RGB')
            if image.size != (224, 224):
                raise ValueError(f'Expected 224x224 image for {row.id_code}; got {image.size}.')
            tensor = self.transform(image)
        return tensor, int(row.diagnosis)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(FundusDataset(train_frame, train_transform), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, worker_init_fn=seed_worker,
                          generator=loader_generator)
validation_loader = DataLoader(FundusDataset(validation_frame, eval_transform), batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=NUM_WORKERS)
train_counts = np.bincount(train_frame.diagnosis.to_numpy(), minlength=5)
if (train_counts == 0).any():
    raise ValueError('Every grade must occur in training to fit the five-grade model.')
class_weights = len(train_frame) / (5.0 * train_counts)
print('Train counts by grade:', train_counts.tolist(), '| class weights:', np.round(class_weights, 3).tolist())

Train counts by grade: [1258, 253, 684, 127, 199] | class weights: [0.401, 1.993, 0.737, 3.97, 2.534]


In [4]:
def collect_predictions(model, loader):
    model.eval()
    truth, predicted = [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            truth.extend(labels.numpy().tolist())
            predicted.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.asarray(truth), np.asarray(predicted)

def metric_summary(truth, predicted):
    labels = list(range(5))
    return {
        'accuracy': float(accuracy_score(truth, predicted)),
        'balanced_accuracy': float(balanced_accuracy_score(truth, predicted)),
        'macro_f1': float(f1_score(truth, predicted, labels=labels, average='macro', zero_division=0)),
        'quadratic_weighted_kappa': float(cohen_kappa_score(truth, predicted, labels=labels, weights='quadratic')),
        'per_class': classification_report(truth, predicted, labels=labels, output_dict=True, zero_division=0),
        'confusion_matrix': confusion_matrix(truth, predicted, labels=labels).tolist(),
        'confusion_matrix_normalized_true': np.nan_to_num(
            confusion_matrix(truth, predicted, labels=labels, normalize='true')
        ).tolist(),
    }

weights = ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 5)
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = OUTPUT_DIR / 'best_resnet18.pt'
best_macro_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss, seen = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        seen += labels.size(0)
    y_true, y_pred = collect_predictions(model, validation_loader)
    validation_metrics = metric_summary(y_true, y_pred)
    row = {'epoch': epoch, 'train_loss': total_loss / seen,
           'validation_macro_f1': validation_metrics['macro_f1'],
           'validation_qwk': validation_metrics['quadratic_weighted_kappa']}
    history.append(row)
    print(row)
    if validation_metrics['macro_f1'] > best_macro_f1:
        best_macro_f1 = validation_metrics['macro_f1']
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch,
                    'validation_metrics': validation_metrics, 'seed': SEED}, checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping after {epoch} epochs; best epoch was {best_epoch}.')
            break

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
y_val, y_val_pred = collect_predictions(model, validation_loader)
validation_results = metric_summary(y_val, y_val_pred)
run_config = {
    'architecture': 'torchvision ResNet-18, ImageNet pretrained, five logits',
    'input': '224x224 RGB; ImageNet mean/std normalization',
    'training_augmentation': 'random horizontal flip and rotation +/-10 degrees',
    'loss': 'training-split inverse-frequency weighted cross-entropy',
    'optimizer': 'AdamW', 'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
    'batch_size': BATCH_SIZE, 'max_epochs': MAX_EPOCHS, 'early_stopping_patience': PATIENCE,
    'selection': 'highest validation macro-F1', 'seed': SEED, 'device': str(DEVICE),
    'data_root': str(DATA_ROOT), 'best_epoch': best_epoch,
}
results_path = OUTPUT_DIR / 'validation_metrics.json'
results_path.write_text(json.dumps({'config': run_config, 'history': history,
                                    'validation': validation_results}, indent=2), encoding='utf-8')
print('Best checkpoint:', checkpoint_path)
print('Validation metrics:', {k: v for k, v in validation_results.items() if k != 'per_class'})
display(pd.DataFrame(validation_results['per_class']).T)
display(pd.DataFrame(validation_results['confusion_matrix_normalized_true'],
                     index=range(5), columns=range(5)))
print('Validation results saved to:', results_path)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/quangduyvu1723/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


{'epoch': 1, 'train_loss': 1.0549338407835192, 'validation_macro_f1': 0.5283418056367518, 'validation_qwk': 0.8276295265798238}
{'epoch': 2, 'train_loss': 0.7913253847354463, 'validation_macro_f1': 0.5531258076258075, 'validation_qwk': 0.8272001468294026}
{'epoch': 3, 'train_loss': 0.6150627377155233, 'validation_macro_f1': 0.5736682788775384, 'validation_qwk': 0.811372299333588}
{'epoch': 4, 'train_loss': 0.5275356685214457, 'validation_macro_f1': 0.6606654122710963, 'validation_qwk': 0.8754018347862758}
{'epoch': 5, 'train_loss': 0.41537737395970514, 'validation_macro_f1': 0.6194803860217, 'validation_qwk': 0.840291778468226}
{'epoch': 6, 'train_loss': 0.36528330589749897, 'validation_macro_f1': 0.5840491511172491, 'validation_qwk': 0.8554046728588804}
{'epoch': 7, 'train_loss': 0.28409788252483614, 'validation_macro_f1': 0.6416563602014904, 'validation_qwk': 0.8696974318032993}
{'epoch': 8, 'train_loss': 0.2774158869646504, 'validation_macro_f1': 0.6041642092406347, 'validation_qwk'

,precision,recall,f1-score,support
0,0.960145,0.981481,0.970696,270.000000
1,0.625000,0.648148,0.636364,54.000000
2,0.727848,0.777027,0.751634,148.000000
3,0.444444,0.461538,0.452830,26.000000
4,0.714286,0.375000,0.491803,40.000000
accuracy,0.821561,0.821561,0.821561,0.821561
macro avg,0.694345,0.648639,0.660665,538.000000
weighted avg,0.819401,0.821561,0.816244,538.000000


,0,1,2,3,4
0,0.981481,0.011111,0.007407,0.000000,0.000000
1,0.092593,0.648148,0.259259,0.000000,0.000000
2,0.033784,0.108108,0.777027,0.067568,0.013514
3,0.038462,0.000000,0.346154,0.461538,0.153846
4,0.000000,0.050000,0.450000,0.125000,0.375000


Validation results saved to: /Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/outputs/aptos_resnet18_run/validation_metrics.json


## Optional test evaluation — run only after review and freeze

The default is locked. Once A and B document review of similar-image candidates, approve the final split, and freeze it, explicitly set `CONFIRM_REVIEWED_FROZEN_SPLIT = True` in the configuration cell and run this cell once. Test results are for the final report only; never use them to choose epochs, thresholds, transforms, or model settings. The default full-notebook run skips test evaluation.

In [5]:
if not CONFIRM_REVIEWED_FROZEN_SPLIT:
    print('Test evaluation skipped: review and freeze the split, then explicitly confirm in the configuration cell.')
elif TEST_EVALUATION_COMPLETED:
    raise RuntimeError('Test evaluation has already been run in this kernel; do not re-evaluate or tune on test.')
else:
    TEST_EVALUATION_COMPLETED = True
    test_loader = DataLoader(FundusDataset(test_frame, eval_transform), batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=NUM_WORKERS)
    y_test, y_test_pred = collect_predictions(model, test_loader)
    test_results = metric_summary(y_test, y_test_pred)
    test_path = OUTPUT_DIR / 'test_metrics.json'
    test_path.write_text(json.dumps({'config': run_config, 'test': test_results}, indent=2), encoding='utf-8')
    print('Final test metrics (do not use for model selection):',
          {k: v for k, v in test_results.items() if k != 'per_class'})
    display(pd.DataFrame(test_results['per_class']).T)
    display(pd.DataFrame(test_results['confusion_matrix_normalized_true'], index=range(5), columns=range(5)))
    print('Test results saved to:', test_path)

Test evaluation skipped: review and freeze the split, then explicitly confirm in the configuration cell.


## Summary and limitations

This notebook defines a reproducible transfer-learning baseline and validation-based checkpoint selection; its outputs become meaningful only after the matching authorized data is available and the notebook is run successfully. Record the handoff version, split-manifest hash, framework/library versions, device, best epoch, and validation metrics with the experiment.

The split is currently provisional: the earlier data audit reported 4,096 near-duplicate candidate pairs that have not been manually resolved, and patient-level independence is not established. Therefore test evaluation remains disabled until the team reviews and freezes the split. A model metric is not evidence of clinical utility, diagnosis correctness, or patient-level generalization. Compare alternatives using validation data only and document every change before the one-time final test evaluation.